In [1]:
from tabpfn_project.paths import DISTNET_DATA_DIR
from tabpfn_project.helper.load_data import load_distnet_data
from tabpfn_project.globals import DISTNET_SCENARIOS
from tabpfn_project.helper.utils import subsample_flattened_data, subsample_features, subsample_targets_per_instance

for scenario in DISTNET_SCENARIOS:
    print(scenario)


clasp_factoring
saps-CVVAR
spear_qcp
yalsat_qcp
spear_swgcp
yalsat_swgcp
lpg-zeno


### NGBoost

In [53]:
import time
import numpy as np

# NGBoost & Scikit-Learn Ecosystem
from ngboost import NGBRegressor
from ngboost.distns import LogNormal
from ngboost.scores import LogScore
from sklearn.tree import DecisionTreeRegressor

from tabpfn_project.globals import RANDOM_STATE
from tabpfn_project.helper.preprocess import delete_constant_features, preprocess_features
from tabpfn_project.helper.scalers import max_scaling

def train_evaluate_ngboost_runtime(X_train_flat: np.ndarray, 
                                   y_train_flat: np.ndarray, 
                                   X_test: np.ndarray, 
                                   y_test: np.ndarray, 
                                   ):
    """
    Trains a robust NGBoost model to predict parametric runtime distributions (LogNormal) 
    for NP-hard combinatorial optimization problems.
    
    Parameters:
    -----------
    X_train_flat : np.ndarray
        Flattened feature matrix of shape (n_instances * n_obs_per_instance, n_features).
    y_train_flat : np.ndarray
        Flattened runtime targets of shape (n_instances * n_obs_per_instance, 1).
    X_test : np.ndarray
        Unflattened feature matrix of shape (n_test_instances, n_features).
    y_test : np.ndarray
        Unflattened runtime targets of shape (n_test_instances, n_obs_per_instance).
        
    Returns:
    --------
    mean_nllh : float
        The Mean Negative Log-Likelihood over the test observations.
    fit_time : float
        Time taken to train the model in seconds.
    test_time : float
        Time taken to run inference and evaluate the model in seconds.
    model : NGBRegressor
        The fitted NGBoost model.
    """
    
    print(f"[INFO] Initializing NGBoost model")

    # 1. default base learner config
    default_tree_learner = DecisionTreeRegressor(
    criterion="friedman_mse",
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_depth=3,
    splitter="best",
    random_state=None,
)

    # 2. NGBoost Architecture Setup
    model = NGBRegressor(
        Dist=LogNormal,             # Targets strictly >0; perfectly models heavy-tail search runtimes
        Score=LogScore,             # Optimizes Maximum Likelihood Estimation (MLE)
        Base=default_tree_learner,  # Default regression tree base learner
        n_estimators=500,          # Ample boosting rounds (relies on early stopping / shrinkage)
        learning_rate=0.01,         # Conservative shrinkage to prevent distributional overfitting
        minibatch_frac=1.0,         # Row subsampling: injects stochasticity and speeds up fitting
        col_sample=1.0,             # Column subsampling: acts as powerful feature regularization
        verbose=True,
        verbose_eval=100,            # Print progress every 100 iterations
        tol=1e-4,
        random_state=RANDOM_STATE,
        validation_fraction=0.1,
        early_stopping_rounds=None,  # None = no early stopping;
    )
    
    X_train_flat, X_test = delete_constant_features(X_train_flat, X_test)
    X_train_flat, X_test = preprocess_features(X_train_flat, X_test)

    y_train_flat_scaled, y_test_scaled, y_scaler = max_scaling(y_train_flat, y_test)

    # Fitting the model
    print("[INFO] Beginning Model Training...")
    start_fit = time.perf_counter()
    model.fit(X_train_flat, y_train_flat_scaled.ravel())
    fit_time = time.perf_counter() - start_fit
    print(f"[INFO] Training Completed in {fit_time:.2f} seconds.")
    
    # 4. Inference & Distributional Evaluation
    print("[INFO] Beginning Inference & Evaluation on Unseen Test Data...")
    start_test = time.perf_counter()
    
    # model.pred_dist outputs a LogNormal object parameterized by \mu and \sigma for each test instance
    test_dists = model.pred_dist(X_test)

    test_time = time.perf_counter() - start_test

    print(f"[INFO] Evaluation Completed in {test_time:.2f} seconds.")
    N_test, T_obs = y_test_scaled.shape
    
    # Array to accumulate the sum of log-likelihoods for each individual test instance
    per_instance_llh_sum = np.zeros(N_test)
    
    # Iterate through each of the T observations
    for t in range(T_obs):
        y_test_t_safe = y_test_scaled[:, t]
        
        # test_dists.logpdf returns an array of shape (N_test,). 
        # We add the t-th observation's log-likelihood to each instance's total.
        per_instance_llh_sum += test_dists.logpdf(y_test_t_safe)
        
    # 1. Calculate per-instance llhs (averaged over the T seeds)
    per_instance_llh_avg = (per_instance_llh_sum / T_obs) + np.log(np.max(y_test_scaled, axis=1))
    
    # 2. Average the Negative Log-Likelihood over the number of all test instances
    mean_nllh = -np.mean(per_instance_llh_avg)
        
    print(f"[RESULT] Test Mean NLLH: {mean_nllh:.4f}")
    
    return mean_nllh, fit_time, test_time, model

### QRF

In [54]:
import time
import numpy as np
from scipy.interpolate import PchipInterpolator
from quantile_forest import RandomForestQuantileRegressor

# TabPFN Project Ecosystem
from tabpfn_project.globals import RANDOM_STATE
from tabpfn_project.helper.preprocess import delete_constant_features, preprocess_features
from tabpfn_project.helper.scalers import max_scaling

def train_evaluate_qrf_runtime(X_train_flat: np.ndarray, 
                               y_train_flat: np.ndarray, 
                               X_test: np.ndarray, 
                               y_test: np.ndarray):
    """
    Trains a robust Quantile Random Forest, transforms the discrete CDF to a continuous PDF 
    using PCHIP interpolation, and calculates the NLLH on the test data.
    """
    print("[INFO] Initializing Quantile Random Forest model")
    
    # 1. QRF Architecture Setup
    model = RandomForestQuantileRegressor(
        n_estimators=1000,
        min_samples_leaf=100,
        random_state=RANDOM_STATE,
        n_jobs=-1                   # Utilize all cores for faster training
    )
    
    print("[INFO] Beginning Model Training...")
    start_fit = time.perf_counter()
    
    # Preprocessing identical to NGBoost reference
    X_train_flat, X_test = delete_constant_features(X_train_flat, X_test)
    X_train_flat, X_test = preprocess_features(X_train_flat, X_test)
    y_train_flat_scaled, y_test_scaled, y_scaler = max_scaling(y_train_flat, y_test)

    # Fitting the model
    model.fit(X_train_flat, y_train_flat_scaled.ravel())
    fit_time = time.perf_counter() - start_fit
    print(f"[INFO] Training Completed in {fit_time:.2f} seconds.")
    
    # -----------------------------------------------------------------
    # 4. Inference & Continuous Distribution Evaluation via SOTA PCHIP
    # -----------------------------------------------------------------
    print("[INFO] Beginning Inference & PCHIP PDF Evaluation on Unseen Test Data...")
    start_test = time.perf_counter()
    
    # Predict dense quantiles (0.001 to 0.999) to construct the CDF
    quantiles = np.arange(0.001, 1.0, 0.001)
    # Convert to standard python list to prevent the library's internal ValueError
    y_pred_quantiles = model.predict(X_test, quantiles=quantiles.tolist())
    print(y_pred_quantiles[0][:10])
    print(y_pred_quantiles[0][-10:])
    print(len(np.unique(y_pred_quantiles[0])))
    
    N_test, T_obs = y_test_scaled.shape
    per_instance_llh_sum = np.zeros(N_test)
    
    for i in range(N_test):
        y_q = y_pred_quantiles[i]
        
        # PCHIP requires strictly increasing x-coordinates.
        # QRF might predict identical runtimes for nearby quantiles (ties).
        # We drop duplicate runtime values to ensure strict monotonicity.
        y_q_unique, unique_indices = np.unique(y_q, return_index=True)
        q_unique = quantiles[unique_indices]
        
        # If the model predicts a pure point mass (only 1 unique value), fallback 
        # to a minimal uniform density to prevent math domain errors.
        if len(y_q_unique) < 2:
            per_instance_llh_sum[i] = T_obs * np.log(1e-6)
            continue
            
        # Fit a monotonic cubic spline to the CDF points (runtime vs quantile)
        cdf_interp = PchipInterpolator(y_q_unique, q_unique, extrapolate=True)
        
        # The analytical derivative of the continuous CDF is our continuous PDF!
        pdf_interp = cdf_interp.derivative()
        
        # Evaluate the continuous PDF exactly at the true observed runtime values
        y_test_obs = y_test_scaled[i]
        pdf_vals = pdf_interp(y_test_obs)
        
        # Clip to prevent log(0) for instances that fall far outside the predicted bounds
        pdf_vals = np.clip(pdf_vals, a_min=1e-10, a_max=None)
        
        # Accumulate the log-likelihoods for this specific instance
        per_instance_llh_sum[i] = np.sum(np.log(pdf_vals))
        
    # Apply your exact NLLH reduction and scaler correction formula
    per_instance_llh_avg = (per_instance_llh_sum / T_obs) + np.log(np.max(y_test_scaled, axis=1))
    mean_nllh = -np.mean(per_instance_llh_avg)
    
    test_time = time.perf_counter() - start_test
    
    print(f"[INFO] Evaluation Completed in {test_time:.2f} seconds.")
    print(f"[RESULT] Test Mean NLLH: {mean_nllh:.4f}")
    
    return mean_nllh, fit_time, test_time, model

### NGBOOST With SMAC3

In [55]:
import time
import numpy as np

# NGBoost & Scikit-Learn Ecosystem
from ngboost import NGBRegressor
from ngboost.distns import LogNormal
from ngboost.scores import LogScore
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

# SMAC3 & ConfigSpace Imports
from ConfigSpace import Configuration, ConfigurationSpace, Float, Integer
from smac import HyperparameterOptimizationFacade as HPOFacade
from smac import Scenario

from tabpfn_project.globals import RANDOM_STATE
from tabpfn_project.helper.preprocess import delete_constant_features, preprocess_features
from tabpfn_project.helper.scalers import max_scaling

def train_evaluate_ngboost(X_train_flat: np.ndarray, 
                            y_train_flat: np.ndarray, 
                            X_test: np.ndarray, 
                            do_hpo: bool,
                            hpo_time: int):
    """
    Trains a robust NGBoost model to predict parametric runtime distributions (LogNormal).
    Optionally performs SMAC3-based Hyperparameter Optimization.
    args:
    X_train_flat : np.ndarray
        Flattened feature matrix of shape (n_instances * n_obs_per_instance, n_features).
    y_train_flat : np.ndarray
        Flattened runtime targets of shape (n_instances * n_obs_per_instance, 1).
    X_test : np.ndarray
        Unflattened feature matrix of shape (n_test_instances, n_features).
    do_hpo : bool
        Whether to perform SMAC3 Hyperparameter Optimization.
    hpo_time : int
        Time limit for HPO in seconds.
    returns the predicted test distributions, best hyperparameters, training time, evaluation time, and y_scaler.
    """
    print(f"[INFO] Initializing NGBoost pipeline")

    # 1. Preprocessing (applied entirely before HPO/Training to maintain consistency)
    X_train_flat, X_test = delete_constant_features(X_train_flat, X_test)
    X_train_flat, X_test = preprocess_features(X_train_flat, X_test)
    y_train_flat_scaled, y_scaler = max_scaling(y_train_flat)
    y_train_eval = y_train_flat_scaled.ravel()

    # Base configuration dictionary (will be overwritten if do_hpo is True)
    best_params = {
        "n_estimators": 500,
        "max_depth": 3,
        "learning_rate": 0.01,
        "minibatch_frac": 1.0,
        "col_sample": 1.0
    }

    # 2. SMAC3 Hyperparameter Optimization
    if do_hpo:
        print(f"[INFO] Starting SMAC3 HPO for NGBoost (Time limit: {hpo_time}s)...")
        # Isolate a validation set specifically for HPO to avoid testing data leakage
        X_tr_hpo, X_val_hpo, y_tr_hpo, y_val_hpo = train_test_split(
            X_train_flat, y_train_eval, test_size=0.15, random_state=RANDOM_STATE
        )

        # Define Search Space
        cs = ConfigurationSpace()
        cs.add([
            Integer("n_estimators", (100, 1500)),
            Integer("max_depth", (2, 5)),
            Float("learning_rate", (0.001, 0.1), log=True),
            Float("minibatch_frac", (0.5, 1.0)),
            Float("col_sample", (0.5, 1.0)),
        ])

        def ngboost_objective(config: Configuration, seed: int) -> float:
            base_tree = DecisionTreeRegressor(
                criterion="friedman_mse", min_samples_split=2, min_samples_leaf=1,
                max_depth=config["max_depth"], splitter="best", random_state=seed
            )
            model = NGBRegressor(
                Dist=LogNormal, Score=LogScore, Base=base_tree,
                n_estimators=config["n_estimators"], # Passed from ConfigSpace directly
                learning_rate=config["learning_rate"],
                minibatch_frac=config["minibatch_frac"], col_sample=config["col_sample"],
                random_state=seed, verbose=False
            )
            # Fit without early stopping. SMAC explicitly evaluates the exact iteration count.
            # This guarantees that SMAC learns the correct interplay between learning_rate and n_estimators.
            model.fit(X_tr_hpo, y_tr_hpo)
            
            # Evaluate via Negative Log-Likelihood (NLL)
            preds = model.pred_dist(X_val_hpo)
            return -preds.logpdf(y_val_hpo).mean()

        scenario = Scenario(configspace=cs, deterministic=True, n_trials=999999, walltime_limit=hpo_time)
        smac = HPOFacade(scenario, ngboost_objective, overwrite=True)
        
        incumbent = smac.optimize()
        best_params = dict(incumbent)
        print(f"[INFO] HPO Completed. Best Config: {best_params}")

    # 3. Final Model Configuration & Training
    print("[INFO] Building Final NGBoost Model...")
    final_base_learner = DecisionTreeRegressor(
        criterion="friedman_mse", min_samples_split=2, min_samples_leaf=1,
        max_depth=best_params["max_depth"], splitter="best", random_state=RANDOM_STATE
    )
    final_model = NGBRegressor(
        Dist=LogNormal, Score=LogScore, Base=final_base_learner,
        n_estimators=best_params["n_estimators"], 
        learning_rate=best_params["learning_rate"],
        minibatch_frac=best_params["minibatch_frac"], col_sample=best_params["col_sample"],
        verbose=True, verbose_eval=100, tol=1e-4, random_state=RANDOM_STATE
    )

    start_fit = time.perf_counter()
    final_model.fit(X_train_flat, y_train_eval)
    fit_time = time.perf_counter() - start_fit
    print(f"[INFO] Final Training Completed in {fit_time:.2f} seconds.")
    
    # 4. Inference & Evaluation
    print("[INFO] Beginning Inference & Evaluation on Unseen Test Data...")
    start_test = time.perf_counter()
    test_dists = final_model.pred_dist(X_test)
    test_time = time.perf_counter() - start_test
    print(f"[INFO] Evaluation Completed in {test_time:.2f} seconds.")
    
    return test_dists, best_params, fit_time, test_time, y_scaler

### QRF With SMAC3

In [ ]:
import time
import numpy as np
from scipy.interpolate import PchipInterpolator
from sklearn.model_selection import train_test_split

# Quantile Forest Ecosystem
from quantile_forest import RandomForestQuantileRegressor

# SMAC3 & ConfigSpace Imports
from ConfigSpace import Configuration, ConfigurationSpace, Float, Integer, Categorical
from smac import HyperparameterOptimizationFacade as HPOFacade
from smac import Scenario

# TabPFN Project Ecosystem
from tabpfn_project.globals import RANDOM_STATE
from tabpfn_project.helper.preprocess import delete_constant_features, preprocess_features
from tabpfn_project.helper.scalers import log_scaling

def train_evaluate_qrf(X_train_flat: np.ndarray, 
                               y_train_flat: np.ndarray, 
                               X_test: np.ndarray, 
                               do_hpo: bool,
                               hpo_time: int,):
    """
    Trains a robust Quantile Random Forest.
    Optionally performs SMAC3-based Hyperparameter Optimization.
    """
    print("[INFO] Initializing Quantile Random Forest pipeline")
    
    # 1. Preprocessing
    X_train_flat, X_test = delete_constant_features(X_train_flat, X_test)
    X_train_flat, X_test = preprocess_features(X_train_flat, X_test)
    y_train_flat_scaled = log_scaling(y_train_flat)
    y_train_eval = y_train_flat_scaled.ravel()

    # Base configuration dictionary (overwritten if do_hpo is True)
    best_params = {
        "n_estimators": 100,
        "min_samples_leaf": 1,
        "max_features": 1.0,
        "max_samples": None,
        "max_depth": None,
        "criterion": "squared_error"
    }

    # 2. SMAC3 Hyperparameter Optimization
    if do_hpo:
        print(f"[INFO] Starting SMAC3 HPO for QRF (Time limit: {hpo_time}s)...")
        # Split data for HPO validation to avoid data leakage
        X_tr_hpo, X_val_hpo, y_tr_hpo, y_val_hpo = train_test_split(
            X_train_flat, y_train_eval, test_size=0.15, random_state=RANDOM_STATE
        )

        cs = ConfigurationSpace()
        cs.add([
            Integer("n_estimators", (100, 1000)),
            Integer("min_samples_leaf", (1, 200)),
            Float("max_features", (0.1, 1.0)),
            Float("max_samples", (0.1, 1.0)),
            Integer("max_depth", (5, 50)),
            Categorical("criterion", ["squared_error", "friedman_mse"])
        ])

        def qrf_objective(config: Configuration, seed: int) -> float:
            model = RandomForestQuantileRegressor(
                n_estimators=config["n_estimators"],
                min_samples_leaf=config["min_samples_leaf"],
                max_features=config["max_features"],
                max_samples=config["max_samples"],
                max_depth=config["max_depth"],
                criterion=config["criterion"],
                bootstrap=True, # Required for max_samples to take effect
                random_state=seed, 
                n_jobs=-1
            )
            model.fit(X_tr_hpo, y_tr_hpo)
            
            # Predict sparse quantiles for fast HPO loss calculation
            hpo_quantiles = np.arange(0.001, 1.0, 0.001).tolist()
            y_pred = model.predict(X_val_hpo, quantiles=hpo_quantiles)
            
            # Compute Average Pinball Loss (Quantile Loss)
            avg_pinball_loss = 0.0
            for i, q in enumerate(hpo_quantiles):
                error = y_val_hpo - y_pred[:, i]
                avg_pinball_loss += np.mean(np.maximum(q * error, (q - 1) * error))
            return avg_pinball_loss / len(hpo_quantiles)

        scenario = Scenario(configspace=cs, deterministic=True, n_trials=999999, walltime_limit=hpo_time)
        smac = HPOFacade(scenario, qrf_objective, overwrite=True)
        
        incumbent = smac.optimize()
        best_params = dict(incumbent)
        print(f"[INFO] HPO Completed. Best Config: {best_params}")

    # 3. Final Model Training
    print("[INFO] Building Final QRF Model...")
    final_model = RandomForestQuantileRegressor(
        n_estimators=best_params["n_estimators"],
        min_samples_leaf=best_params["min_samples_leaf"],
        max_features=best_params["max_features"],
        max_samples=best_params["max_samples"],
        max_depth=best_params["max_depth"],
        criterion=best_params["criterion"],
        bootstrap=True,
        random_state=RANDOM_STATE, 
        n_jobs=-1
    )
    
    start_fit = time.perf_counter()
    final_model.fit(X_train_flat, y_train_eval)
    fit_time = time.perf_counter() - start_fit
    print(f"[INFO] Final Training Completed in {fit_time:.2f} seconds.")
    
    # 4. Inference & Evaluation
    print("[INFO] Beginning Inference & Evaluation on Unseen Test Data...")
    start_test = time.perf_counter()
    
    # Predict dense quantiles (0.001 to 0.999) to construct the CDF
    quantiles = np.arange(0.001, 1.0, 0.001)
    y_pred_quantiles = final_model.predict(X_test, quantiles=quantiles.tolist())
    test_time = time.perf_counter() - start_test
    

    print(f"[INFO] Evaluation Completed in {test_time:.2f} seconds.")

    return y_pred_quantiles, best_params, fit_time, test_time

## Testing

In [2]:
import pickle
import platform

import torch

from tabpfn_project.helper.tree_based_models import calculate_all_distribution_metrics_ngboost_logspace


scenario = DISTNET_SCENARIOS[0]
fold = 0
context_size = 2**15
seed = 100
*_, y_test = load_distnet_data(DISTNET_DATA_DIR, scenario, fold)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from tabpfn_project.helper.utils import WindowsPathUnpickler
from tabpfn_project.paths import RESULTS_DIR

ngboost_path = RESULTS_DIR / "experiment_context_sizes" / f"ngboost_{scenario}.pkl"

with open(ngboost_path, 'rb') as f:
    # Use our custom Unpickler on Windows, otherwise standard pickle
    if platform.system() == 'Windows':
        results_dict = WindowsPathUnpickler(f).load()
    else:
        results_dict = pickle.load(f)

ngboost_instance_dict = results_dict['ngboost'][scenario][context_size][fold][seed]

qrf_path = RESULTS_DIR / "experiment_context_sizes" / f"qrf_{scenario}.pkl"

with open(qrf_path, 'rb') as f:
    # Use our custom Unpickler on Windows, otherwise standard pickle
    if platform.system() == 'Windows':
        results_dict = WindowsPathUnpickler(f).load()
    else:
        results_dict = pickle.load(f)
qrf_instance_dict = results_dict['qrf'][scenario][context_size][fold][seed]
# experiment_results_dict \
#             .setdefault(model_name, {}) \
#             .setdefault(scenario, {}) \
#             .setdefault(context_size, {}) \
#             .setdefault(fold, {})[seed_context] = results_dict

from tabpfn_project.helper.tree_based_models import calculate_all_distribution_metrics_qrf_logspace_pchip, calculate_all_distribution_metrics_qrf_logspace_kde


metrics_summary_pchip, _ = calculate_all_distribution_metrics_qrf_logspace_pchip(y_test, qrf_instance_dict['test_preds'], device=device, y_scaler=None, N_grid_points=15000)
metrics_summary_kde, _ = calculate_all_distribution_metrics_qrf_logspace_kde(y_test, qrf_instance_dict['test_preds'], device=device, y_scaler=None, N_grid_points=15000)

print(f"Metrics Summary (PCHIP): {metrics_summary_pchip.items()}")
print(f"Metrics Summary (KDE): {metrics_summary_kde.items()}")


Train data loaded
Test data loaded
(2000, 113) (2000, 100)
Discarding 0 (2000) instances because of CRASHED
Discarding 0 (2000) instances because of TIMEOUT
Discarding 0 (2000) instances because not stated TIMEOUTS
Discarding 0 (2000) instances because of constant features
Discarding 0 (2000) instances because of UNSAT
Imputed 149 values with median
 bias.shape: torch.Size([200]), nlog_pdf.shape: torch.Size([200, 100])
Metrics Summary (PCHIP): dict_items([('NLLH_mean', 0.8624182939529419), ('NLLH_std', 1.8486571311950684), ('CRPS_mean', 0.006063112989068031), ('CRPS_std', 0.001983637921512127), ('Wasserstein_mean', 0.004398110322654247), ('Wasserstein_std', 0.0026459351647645235), ('KS_mean', 0.24252299964427948), ('KS_std', 0.10274314880371094)])
Metrics Summary (KDE): dict_items([('NLLH_mean', -0.19438989460468292), ('NLLH_std', 0.18085455894470215), ('CRPS_mean', 0.006063811015337706), ('CRPS_std', 0.0019894675351679325), ('Wasserstein_mean', 0.004397147800773382), ('Wasserstein_std

Metrics Summary (PCHIP): dict_items([('NLLH_mean', 9.97318172454834), ('NLLH_std', 14.240833282470703), ('CRPS_mean', 0.006785569246858358), ('CRPS_std', 0.0025134659372270107), ('Wasserstein_mean', 0.00624668225646019), ('Wasserstein_std', 0.0037500534672290087), ('KS_mean', 0.32599306106567383), ('KS_std', 0.11647491902112961)])
Metrics Summary (KDE): dict_items([('NLLH_mean', -0.06438225507736206), ('NLLH_std', 0.23237687349319458), ('CRPS_mean', 0.006782069802284241), ('CRPS_std', 0.0025136591866612434), ('Wasserstein_mean', 0.00623335549607873), ('Wasserstein_std', 0.0037420957814902067), ('KS_mean', 0.325276643037796), ('KS_std', 0.115737184882164)])

In [ ]:
# metrics_summary, instance_summary = calculate_all_distribution_metrics_ngboost_logspace(y_test, ngboost_instance_dict['test_preds'], device=device, y_scaler=ngboost_instance_dict['y_scale'], N_grid_points=15000)

# print(metrics_summary.items())

dict_items([('NLLH_mean', -0.19794948399066925), ('NLLH_std', 0.22117206454277039), ('CRPS_mean', 0.006085749715566635), ('CRPS_std', 0.0021450980566442013), ('Wasserstein_mean', 0.004566865973174572), ('Wasserstein_std', 0.0028887458611279726), ('KS_mean', 0.23293332755565643), ('KS_std', 0.1256546676158905)])


dict_items([('NLLH_mean', -0.061780527234077454), ('NLLH_std', 0.4075416922569275), ('CRPS_mean', 0.006409803405404091), ('CRPS_std', 0.0024766339920461178), ('Wasserstein_mean', 0.004984121769666672), ('Wasserstein_std', 0.003332330146804452), ('KS_mean', 0.28154483437538147), ('KS_std', 0.1380416750907898)])


nlog_pdf.shape: torch.Size([200, 100])
dict_items([('NLLH_mean', -0.198185533285141), ('NLLH_std', 0.23036357760429382), ('CRPS_mean', 0.0060987710021436214), ('CRPS_std', 0.0021577731240540743), ('Wasserstein_mean', 0.004541270900517702), ('Wasserstein_std', 0.002915411489084363), ('KS_mean', 0.23487941920757294), ('KS_std', 0.1268477439880371)])

In [15]:
import numpy as np
-np.log(1e-300)

np.float64(690.7755278982137)